# Table 1 benchmark significance (reproducer)

Reproduces the significance tests behind main-text **Table 1** from the committed, leak-audited
artifacts. Supersedes the earlier draft (`08_table1_auc_significance_tests.ipynb`), which predated
the Jul-2026 principled regen: the feature scopes are now leak-free on **both** sides (our biology
excludes trial-design/context; their design excludes the enrollment leak + establishment proxies),
and inClinico is treated as an inflation exhibit, not a head-to-head.

Sources of truth: `scripts/benchmark/benchmark_final_v2.py` -> `results/benchmark/benchmark_final_v2.json`,
plus the per-fold CSVs it consumes. The two head-to-heads (HINT, TrialBench) are **paired** by
(seed, fold) on shared splits, so a paired *t* / Wilcoxon is the correct test; DeLong is the
trial-level equivalent where paired per-trial predictions exist.

In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
B = ROOT / "results" / "benchmark"
FIN = json.load(open(B / "benchmark_final_v2.json"))
print("locked numbers:", json.dumps({k: (v if k=="inclinico" else "...") for k,v in FIN.items()}, indent=2)[:400])

## 1. HINT (fair, structure-holdout): paired tests on the 25 (seed, fold) AUCs

In [ ]:
star = pd.read_csv(B/"star_phase3_split_compare_clean.csv")
star = star[star.arm=="holdout"][["seed","fold","roc_auc"]].rename(columns={"roc_auc":"ours"})
hint = pd.read_csv(B/"hint_holdout_clean_full.csv")[["seed","fold","roc_auc"]].rename(columns={"roc_auc":"hint"})
p = star.merge(hint, on=["seed","fold"])
t, w = stats.ttest_rel(p.ours, p.hint), stats.wilcoxon(p.ours, p.hint)
print(f"HINT fair: ours {p.ours.mean():.3f} vs HINT {p.hint.mean():.3f} | +{p.ours.mean()-p.hint.mean():.3f} | "
      f"{(p.ours>p.hint).sum()}/25 | paired t p={t.pvalue:.1e}, Wilcoxon p={w.pvalue:.1e}")

### 1b. DeLong on the seed-0 per-trial predictions (corrected)

The earlier draft merged the two per-trial files on `nctid` alone; 155 NCTs carry two indication
rows, so that many-to-many join inflated N to 3,316 and violated DeLong's independence assumption.
Here we aggregate to one prediction per unique NCT (mean over indication rows) before the test.

In [ ]:
def midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2
def delong(y,a,b):
    y=np.asarray(y); o=(-y).argsort(kind="mergesort"); n1=int(y.sum()); n=len(y)-n1
    P=np.vstack([np.asarray(a)[o],np.asarray(b)[o]])
    tx=np.array([midrank(P[r,:n1]) for r in range(2)]); ty=np.array([midrank(P[r,n1:]) for r in range(2)])
    tz=np.array([midrank(P[r]) for r in range(2)])
    auc=(tz[:,:n1].sum(1)/n1-(n1+1)/2)/n
    v01=(tz[:,:n1]-tx)/n; v10=1-(tz[:,n1:]-ty)/n1
    S=np.cov(v01)/n1+np.cov(v10)/n; L=np.array([[1,-1]]); var=(L@S@L.T)[0,0]
    z=(auc[0]-auc[1])/np.sqrt(var); return float(auc[0]),float(auc[1]),float(z),float(2*stats.norm.sf(abs(z)))
sp=pd.read_csv(B/"star_pertrial_seed0.csv").groupby("nct").agg(label=("label","first"),p=("proba","mean")).reset_index()
hp=pd.read_csv(B/"hint_pertrial_seed0.csv").groupby("nctid").agg(label=("label","first"),p=("pred","mean")).reset_index().rename(columns={"nctid":"nct"})
m=sp.merge(hp,on="nct",suffixes=("_s","_h")); assert (m.label_s==m.label_h).all()
am,ah,z,pv=delong(m.label_s.values,m.p_s.values,m.p_h.values)
print(f"DeLong (seed 0, {len(m)} UNIQUE trials): ours {am:.3f} vs HINT {ah:.3f} | z={z:.2f}, p={pv:.1e}")

## 2. TrialBench (fair): our biology vs their legitimate trial-design

Both feature sets are cross-validated on the same joined cohort with shared (seed, fold) splits;
`benchmark_final_v2.py` records the paired test on the 15 aligned folds.

In [ ]:
fb = FIN["trialbench"]["fair_biology_vs_legit_design"]
print(f"TrialBench fair: our biology {fb['mean_a']:.3f} vs their legit design {fb['mean_b']:.3f} | "
      f"+{fb['delta']:.3f} | {fb['wins']}/{fb['n']} | paired t p={fb['ttest_p']:.1e}, Wilcoxon p={fb['wilcoxon_p']:.1e}")
tb=FIN["trialbench"]["auc"]
print(f"  ladder: their all {tb['their_all']:.3f} -> minus enrollment {tb['their_minus_enrollment']:.3f} "
      f"-> minus establishment {tb['their_legit_design']:.3f} | combined ours+theirs {tb['combined_ours_plus_theirs']:.3f}")

## 3. inClinico: supporting exhibit (**not** in Table 1, not a head-to-head)

inClinico's model is not public, so this is a same-class reconstruction of its Phase 2->3 transition
*task*, not a comparison against their model. It is **deliberately absent from Table 1**, which carries
only the two clean head-to-heads (HINT, TrialBench); inClinico appears as one Discussion sentence plus
Supplementary Figure S3 and Supplementary Note S4.

What the ladder shows: their 0.88 reproduces as a random-CV number (0.875). Under an honest
structure-holdout split the same full modality set scores 0.746 -- and that residual is **trial and
sponsor detail** (0.744 from design + eligibility + sponsor + facility + `n_phase2_trials` alone),
not the compound. Compound biology alone is ~chance (0.545). So the task scores a sponsor's
development decision, not the molecule.

Do **not** restate this as "the benchmark collapses to chance" (an earlier overclaim, retracted): the
honest number is 0.746, carried by establishment/trial detail. Only the *biology* arm is at chance.

In [ ]:
ic = FIN["inclinico"]
print(f"reported {ic['reported_inclinico']} | reproduced random-CV {ic['reproduced_gcv_full']:.3f}")
print(f"  honest structure-holdout, full modality set : {ic['honest_full']:.3f}")
print(f"    of which trial/sponsor detail alone       : {ic['honest_trial_sponsor_only']:.3f}  <- carries it")
print(f"    compound precedent alone                  : {ic['honest_compound_precedent_only']:.3f}")
print(f"    compound biology alone                    : {ic['honest_biology_only']:.3f}  <- ~chance")
print(f"  n_phase2_trials univariate AUC              : {ic['nphase2_univariate_auc']:.2f}  (future-peeking count)")

## Summary

**Table 1** carries the two clean head-to-heads only:

| Benchmark | Fair (leak-free) | Significance | On their terms |
|---|---|---|---|
| HINT | 0.704 vs 0.626 | paired *t* P=7e-9; Wilcoxon P=2e-7; DeLong P=3e-9 | 0.737 vs 0.654 |
| TrialBench | biology 0.768 vs design 0.637 | paired *t* P=1e-6; Wilcoxon P=6e-5 | 0.807 vs 0.794 |

inClinico is **not** a Table 1 row (see section 3): reproduces 0.875 ≈ 0.88 under random CV, 0.746
honest, carried by trial/sponsor detail (0.744) with biology at chance (0.545).

The next cell asserts every one of these against the manuscript text, so this notebook fails loudly
if an artifact is regenerated and a number moves.

In [ ]:
# Guard: every benchmark number the manuscript states, asserted against the locked artifacts.
# If a rebuild moves one of these, this cell fails and the manuscript text must be updated.
tb, tbf, hf, hb, dl = (FIN["trialbench"]["auc"], FIN["trialbench"]["fair_biology_vs_legit_design"],
                       FIN["hint"]["fair_holdout"], FIN["hint"]["their_turf_blind"],
                       FIN["hint"]["delong_pertrial_seed0"])
est = FIN["trialbench"]["establishment_block_auc"]
split = pd.read_csv(B / "trialbench_split_compare.csv").set_index("feature_set")

# The manuscript cites AUCs to 3 dp, so an artifact agrees if it sits within half a unit of the
# last cited place. Do NOT test round(artifact, 3) == cited: HINT's 0.7045 is an exact rounding
# midpoint, where the tie-break direction is a float artifact rather than a real disagreement.
TOL = 5e-4 + 1e-9  # half the last cited digit, plus float slack for exact midpoints

checks = [  # (manuscript location, cited value, artifact value)
    ("L44/T1 TrialBench their_all",        0.794, tb["their_all"]),
    ("L44 minus-enrollment",               0.738, tb["their_minus_enrollment"]),
    ("L44/T1 legit design (honest)",       0.637, tb["their_legit_design"]),
    ("L44/T1 our biology",                 0.768, tb["our_biology"]),
    ("L44 combined",                       0.807, tb["combined_ours_plus_theirs"]),
    ("T1 TrialBench delta",                0.131, tbf["delta"]),
    ("L44 establishment floor (min)",      0.591, min(est["industry_sponsorship"],
                                                      est["data_monitoring_committee"],
                                                      est["fda_regulated_status"])),
    ("L44 establishment ceiling (max)",    0.626, max(est["industry_sponsorship"],
                                                      est["data_monitoring_committee"],
                                                      est["fda_regulated_status"])),
    ("L42/T1 HINT ours",                   0.704, hf["mean_a"]),
    ("L42/T1 HINT theirs",                 0.626, hf["mean_b"]),
    ("L42 HINT delta",                     0.078, hf["delta"]),
    ("L42 HINT blind ours",                0.737, hb["mean_a"]),
    ("L42 HINT blind theirs",              0.654, hb["mean_b"]),
    ("FigS2a ours holdout",                0.768, split.loc["our biology", "holdout_auc"]),
    ("FigS2a ours blind",                  0.842, split.loc["our biology", "blind_auc"]),
    ("FigS2a TrialBench holdout",          0.637, split.loc["their legit design", "holdout_auc"]),
    ("FigS2a TrialBench blind",            0.652, split.loc["their legit design", "blind_auc"]),
]
counts = [("T1 cohort: TrialBench trials", 1142, FIN["trialbench"]["n_trials"]),
          ("T1 cohort: TrialBench compounds", 412, FIN["trialbench"]["n_compounds"]),
          ("T1 HINT wins", 24, hf["wins"]), ("T1 TrialBench wins", 15, tbf["wins"]),
          ("Methods HINT shared trials", 2787, dl["n"])]
pvals = [("T1 TrialBench paired t", 1e-6, tbf["ttest_p"]), ("T1 TrialBench Wilcoxon", 6e-5, tbf["wilcoxon_p"]),
         ("T1 HINT paired t", 7e-9, hf["ttest_p"]), ("T1 HINT Wilcoxon", 2e-7, hf["wilcoxon_p"]),
         ("T1 HINT DeLong", 3e-9, dl["delong_p"])]

bad = []
for what, cited, actual in checks:
    ok = abs(cited - actual) <= TOL
    bad += [] if ok else [f"{what}: manuscript {cited} vs artifact {actual:.4f}"]
    print(f"{'OK ' if ok else 'FAIL'} {what:38s} cited {cited:.3f} | artifact {actual:.4f}")
for what, cited, actual in counts:
    ok = cited == actual
    bad += [] if ok else [f"{what}: manuscript {cited} vs artifact {actual}"]
    print(f"{'OK ' if ok else 'FAIL'} {what:38s} cited {cited}     | artifact {actual}")
for what, cited, actual in pvals:  # p-values are cited to one significant figure
    ok = f"{actual:.0e}" == f"{cited:.0e}"
    bad += [] if ok else [f"{what}: manuscript {cited:.0e} vs artifact {actual:.1e}"]
    print(f"{'OK ' if ok else 'FAIL'} {what:38s} cited {cited:.0e}   | artifact {actual:.1e}")
assert not bad, "MANUSCRIPT/ARTIFACT MISMATCH:\n  " + "\n  ".join(bad)
print(f"\nAll {len(checks)+len(counts)+len(pvals)} manuscript benchmark numbers reproduce.")